In [ ]:
import sys
lib_path = [r'C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisRoutine',
            r'C:\Users\ikahb\OneDrive\Applications\GitHub\SeisRoutine']
for path in lib_path:
    sys.path.append(path)
##########################################################################
import SeisRoutine.catalog as src
import SeisRoutine.waveform as srw
import SeisRoutine.waveform.health_check.spike as spike_checker
import SeisRoutine.config as srconf
import SeisRoutine.statistics as srs

In [ ]:
import seisbench.data as sbd
import seisbench.generate as sbg
import numpy as np
import os
from scipy import signal
from tqdm import tqdm
from scipy.stats import skew
import matplotlib.pyplot as plt

In [ ]:
from obspy.signal.filter import bandpass

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
import pandas as pd

def df2heatmap(conditions, title=None):
    true_percent = conditions.mean() * 100
    #
    summary = pd.DataFrame({"True %": true_percent,
                            "False %": 100 -true_percent})
    print(summary)
    # ساخت label جدید
    labels = [
        f"{col}\n({true_percent[col]:.1f}%)"
        for col in conditions.columns
    ]
    plt.figure(figsize=(12, 6))
    cmap = ListedColormap(["#beaed4", "#7fc97f"]) # [flase, true]
    cmap = ListedColormap([
        # "#fc8d62",
        # "#66c2a5",
        "#66c2a5",
        "#fc8d62",
    ])
    # cmap='gray_r'
    img = plt.imshow(
        conditions.T,
        aspect='auto',
        cmap=cmap,
        interpolation='nearest'
    )
    plt.gca().set_yticks(
        np.arange(-0.5, conditions.shape[1], 1),
        minor=True,
    )
    plt.grid(which='minor', color='white', linestyle='-', linewidth=2)
    plt.yticks(
        range(len(labels)),
        labels
    )
    plt.xlabel("Sample Index")
    plt.ylabel("Conditions")
    plt.title(title)
    # plt.colorbar(label="Condition")
    cbar = plt.colorbar(img, ticks=[0, 1], label="Condition")
    cbar.ax.set_yticklabels(["False", "True"])

    plt.show()

In [ ]:
file_path = r'c:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisPhaseTune\Training\Configs\Parameters-cfg.yml'
timestamp = srconf.timestamp()
cfg = srconf.Config.load(
    file_path=file_path,
    resolve=True,
)
context={
    "timestamp": timestamp,
}
cfg.resolve(context=context)

In [ ]:
for cha in "ENZ":
    conditions = metadata[[key for key in metadata.keys() if key.startswith(f'trace_{cha}_spike')]]
    metadata[f'all_{cha}'] = conditions.sum(axis=1) >= 4
    conditions = conditions.dropna()
    conditions = conditions.astype(int)
    df2heatmap(conditions, title=None)
# conditions.iloc[1]

In [ ]:
# for idx in range(50, 100):
#     sample = generator[idx]
#     data_3c = sample['X']
#     plt.plot(data_3c[:, :].T + [2, 0, -2], label=['Z', 'N', 'E'])
#     plt.title(str(idx))
#     plt.legend()
#     plt.show()
#     if idx > 100:
#         break

In [ ]:
# [key for key, val in features.items() if ('spike' in key) and (val)]

In [ ]:
# sample = generator[77]
# data_3c = sample['X']
# # plt.plot(data_3c[:, 400:].T + [2, 0, -2], label=['Z', 'N', 'E'])
# # plt.title(str(idx))
# # plt.legend()
# # plt.show()
# c = data_3c[2, :]
# (c.max(),
#  c.min(),
#  spike_checker.spike_by_skewness(c, threshold=3, axis=0),
#  spike_checker.spike_by_kurtosis(c, threshold=100, axis=0, preprocessing=False),
#  min_max_ratio(c),
# )

In [ ]:
# msk = metadata[['all_E', 'all_N', 'all_Z']].sum(axis=1) != 0
# msk = metadata[[f"trace_{cha}_spike: min_max_ration" for cha in "ENZ"] +
#                [f"trace_{cha}_spike: skewness" for cha in "ENZ"]
#                ].sum(axis=1) != 0
# # metadata[msk]

In [ ]:
for idx, features in metadata[msk].iterrows():
    sample = generator[idx]
    data_3c = sample['X']
    label = [key for key, val in features.items() if ('spike' in key) and val]
    label = '\n'.join(label)
    plt.plot(data_3c[:, 400:].T + [2, 0, -2], label=['Z', 'N', 'E'])
    plt.title(str(idx)+label)
    plt.legend()
    plt.show()
    if idx > 100:
        break

In [ ]:
from myfuncs.spike_detection import detect_spikes_ensemble

In [ ]:
conditions[conditions['all']==1].index

In [ ]:
keys = [
    # 'trace_E_spike: zscore',
    # 'trace_E_spike: mad',
    # 'trace_E_spike: wavelet',
    'trace_E_spike: skewness',
    'trace_N_spike: skewness',
    'trace_Z_spike: skewness',
]
msk = metadata[keys].any(axis=1)
for idx, row in metadata[msk].iterrows():
    sample = generator[idx]
    data_3c = sample['X']
    plt.plot(data_3c[:, 400:1000].T + [-2, 0, 2])
    for data_1c in data_3c:
        spike_mask, vote_count = detect_spikes_ensemble(
            data_1c,
            sampling_rate=100.0,
            vote_threshold=3,
        )
        print(spike_mask.any(), vote_count.max())
    plt.title(str(idx))
    plt.show()
    if idx > 100:
        break